In [ ]:
import os
import mne
import warnings
import scipy.signal as signal

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from typing import List, Dict, Tuple
import import_ipynb
import eeg_preprocessing_interface as eeg_pp
import eeg_band_separation as eeg_band
import eeg_complexity_feture as eeg_feture
from tqdm import tqdm
from scipy.io import savemat 

mne.set_log_level("ERROR")
warnings.filterwarnings(
    "ignore",  # Action: ignore warnings
    category=RuntimeWarning,  # Warning type: RuntimeWarning
    message="This filename.*does not conform to MNE naming conventions"  # Warning pattern (regex match)
)
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="NOTE: pick_channels\\(\\) is a legacy function. New code should use inst.pick\\(\\.\\.\\.\\)."
)

In [ ]:
# 1. Basic Configuration (Subjects, Directories, Time Points, etc.)
sub_10hz = ["100102", "100203", "100306", "100412", "100515", "100617", "100723", "100927", "101029", "101139",
            "101449", "101551", "101656", "101758", "101861", "101965", "102274", "102375", "102478", "102580"]

sub_sham = ["300104", "300207", "300308", "300409", "300618", "300720","300822", "300925", "301041", "301250", 
            "301354", "301455", "301560", "301662", "301766", "301867", "302177", "302282", "302384"]  # Removed "300513"

# Directory Configuration
base_dir_10hz = r'D:\山东第一医科大学\数据\Task\八因子\脑网络\10hz'
base_dir_sham = r'D:\山东第一医科大学\数据\Task\八因子\脑网络\sham'
out_base_dir = r'D:\山东第一医科大学\数据\Task\八因子\二值化'  # Can be used for saving results later if needed
target_bands = ['delta', 'theta', 'alpha', 'lbeta', 'hbeta', 'gamma1', 'gamma2', 'gamma3', 'gamma4', 'gamma5', 'gamma6']

# Experiment Dimension Configuration
time_points = ['post','pre'] 
emotions = ['happy','sad']
groups = {
    '10hz': {'subjects': sub_10hz, 'source_dir': base_dir_10hz},
    'sham': {'subjects': sub_sham, 'source_dir': base_dir_sham}
}
extractor = eeg_feture.EEGComplexityAnalyzer()

In [ ]:
def find_threshold(plv_matrix):
    # Binarize matrix using 0.3 as threshold
    binary_matrix = np.where(plv_matrix >= 0.3, 1, 0)
    # Force diagonal to 0 (remove self-connections)
    np.fill_diagonal(binary_matrix, 0)
    
    return binary_matrix

In [ ]:
# Add progress bar for group loop
for group_name, group_info in tqdm(groups.items(), desc="Processing Groups", unit="group"):
    source_dir = group_info['source_dir']
    subject_list = group_info['subjects']  # 20 subjects
    
    # Add progress bar for time point loop
    for time in tqdm(time_points, desc=f"Processing {group_name} Time Points", unit="time point", leave=False):
        # Add progress bar for emotion loop
        for emotion in tqdm(emotions, desc=f"Processing {time} Emotions", unit="emotion", leave=False):
            sub_file = []
            # 3. Iterate over subjects with progress bar
            for sub_idx, sub in enumerate(tqdm(subject_list, desc="Processing Subjects", unit="subject", leave=False)):
                
                # Iterate over frequency bands
                band_file = [] 
                for band_idx, band_name in enumerate(target_bands):
                    # Load file path
                    sub_source_dir = os.path.join(source_dir, time, emotion, sub, f"{band_name}.npy")
                    # Check if file exists
                    if not os.path.exists(sub_source_dir):
                        print(f"Warning: File not found, skipping -> {sub_source_dir}")
                        continue  # Skip missing files and continue
                    
                    matrix = np.load(sub_source_dir)
                    binary_matrix = find_threshold(matrix)
                    band_file.append(binary_matrix)    
                sub_file.append(band_file) 
       
            filename = f'{group_name}_{time}_{emotion}.npy'
            outpath = os.path.join(out_base_dir, filename)
            np.save(outpath, np.array(sub_file))
            print(np.array(sub_file).shape)